## SRP017457

**paper:** [no paper]() - 

**date, curator:** 2026-05-07, Sara Carsanaro

**notes**
* there is no paper but lots of information in SRA

### annotation summary

In [23]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,liver,UBERON:0002107,liver,perfect match


In [24]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adult,UBERON:0000113,post-juvenile adult stage,perfect match
11,3 mo old,UBERON:0000112,sexually immature stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP017457"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 16/16 [00:16<00:00,  1.01s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX209128,SRP017457,Illumina HiSeq 2000,SRS378977,,,,,,liver,Adult,,,,,,,158814,,,,,,Generic sample from Terrapene carolina,SAMN01823446,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Box Turtle,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX209127,SRP017457,Illumina HiSeq 2000,SRS378976,,,,,,liver,Adult,,,,,,,367368,,,,,,Generic sample from Pelusios castaneus,SAMN01823445,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sideneck turtle,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX209126,SRP017457,Illumina HiSeq 2000,SRS378975,,,,,,liver,Adult,,,,,,,142480,,,,,,Generic sample from Sternotherus odoratus,SAMN01823444,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Stinkpot turtle,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX209125,SRP017457,Illumina HiSeq 2000,SRS378974,,,,,,liver,Adult,,,,,,,8475,,,,,,Generic sample from Chelydra serpentina,SAMN01823443,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Snapping turtle,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX209124,SRP017457,Illumina HiSeq 2000,SRS378972,,,,,,liver,Adult,,,,,,,51853,,,,,,Generic sample from Candoia aspera,SAMN01823442,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Viper boa,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX209123,SRP017457,Illumina HiSeq 2000,SRS378973,,,,,,liver,Adult,,,,,,,196253,,,,,,Generic sample from Xenopeltis unicolor,SAMN01823441,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sunbeam Snake,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX209122,SRP017457,Illumina HiSeq 2000,SRS378971,,,,,,liver,Adult,,,,,,,8715,,,,,,Generic sample from Agkistrodon piscivorus,SAMN01823440,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Cottonmouth snake,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX209121,SRP017457,Illumina HiSeq 2000,SRS378970,,,,,,liver,Adult,,,,,,,231986,,,,,,Generic sample from Lasius fuliginosus,SAMN01823439,,,,,,,,,,,07/05/2026,,,African House Snake,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX209120,SRP017457,Illumina HiSeq 2000,SRS378969,,,,,,liver,Adult,,,,,,,481883,,,,,,Generic sample from Eublepharis macularius,SAMN01823438,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_1

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['liver']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0002107'
library.loc[:,'anatName'] = 'liver'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX209128,SRP017457,Illumina HiSeq 2000,SRS378977,UBERON:0002107,liver,,,,liver,Adult,perfect match,not documented,,,,,158814,,,,,,Generic sample from Terrapene carolina,SAMN01823446,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Box Turtle,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX209127,SRP017457,Illumina HiSeq 2000,SRS378976,UBERON:0002107,liver,,,,liver,Adult,perfect match,not documented,,,,,367368,,,,,,Generic sample from Pelusios castaneus,SAMN01823445,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sideneck turtle,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX209126,SRP017457,Illumina HiSeq 2000,SRS378975,UBERON:0002107,liver,,,,liver,Adult,perfect match,not documented,,,,,142480,,,,,,Generic sample from Sternotherus odoratus,SAMN01823444,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Stinkpot turtle,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX209125,SRP017457,Illumina HiSeq 2000,SRS378974,UBERON:0002107,liver,,,,liver,Adult,perfect match,not documented,,,,,8475,,,,,,Generic sample from Chelydra serpentina,SAMN01823443,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Snapping turtle,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX209124,SRP017457,Illumina HiSeq 2000,SRS378972,UBERON:0002107,liver,,,,liver,Adult,perfect match,not documented,,,,,51853,,,,,,Generic sample from Candoia aspera,SAMN01823442,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Viper boa,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX209123,SRP017457,Illumina HiSeq 2000,SRS378973,UBERON:0002107,liver,,,,liver,Adult,perfect match,not documented,,,,,196253,,,,,,Generic sample from Xenopeltis unicolor,SAMN01823441,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sunbeam Snake,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX209122,SRP017457,Illumina HiSeq 2000,SRS378971,UBERON:0002107,liver,,,,liver,Adult,perfect match,not documented,,,,,8715,,,,,,Generic sample from Agkistrodon piscivorus,SAMN01823440,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Cottonmouth snake,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX209121,SRP017457,Illumina HiSeq 2000,SRS378970,UBERON:0002107,liver,,,,liver,Adult,perfect match,not documented,,,,,231986,,,,,,Generic sample from Lasius fuliginosus,SAMN01823439

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['3 mo old' 'Adult']


In [8]:
# conditional (based off infoStage)
library.loc[library["infoStage"] == "Adult", "stageId"] = "UBERON:0000113"
library.loc[library["infoStage"] == "Adult", "stageName"] = "post-juvenile adult stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Adult", "stageAnnotationStatus"] = "perfect match"

# conditional (based off infoStage)
library.loc[library["infoStage"] == "3 mo old", "stageId"] = "UBERON:0000112"
library.loc[library["infoStage"] == "3 mo old", "stageName"] = "sexually immature stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "3 mo old", "stageAnnotationStatus"] = "other"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX209128,SRP017457,Illumina HiSeq 2000,SRS378977,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,,,,158814,,,,,,Generic sample from Terrapene carolina,SAMN01823446,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Box Turtle,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX209127,SRP017457,Illumina HiSeq 2000,SRS378976,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,,,,367368,,,,,,Generic sample from Pelusios castaneus,SAMN01823445,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sideneck turtle,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX209126,SRP017457,Illumina HiSeq 2000,SRS378975,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,,,,142480,,,,,,Generic sample from Sternotherus odoratus,SAMN01823444,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Stinkpot turtle,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX209125,SRP017457,Illumina HiSeq 2000,SRS378974,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,,,,8475,,,,,,Generic sample from Chelydra serpentina,SAMN01823443,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Snapping turtle,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX209124,SRP017457,Illumina HiSeq 2000,SRS378972,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,,,,51853,,,,,,Generic sample from Candoia aspera,SAMN01823442,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Viper boa,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX209123,SRP017457,Illumina HiSeq 2000,SRS378973,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,,,,196253,,,,,,Generic sample from Xenopeltis unicolor,SAMN01823441,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sunbeam Snake,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX209122,SRP017457,Illumina HiSeq 2000,SRS378971,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,,,,8715,,,,,,Generic sample from Agkistrodon piscivorus,SAMN01823440,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry.

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [9]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX209128,SRP017457,Illumina HiSeq 2000,SRS378977,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,158814,,,,,,Generic sample from Terrapene carolina,SAMN01823446,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Box Turtle,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX209127,SRP017457,Illumina HiSeq 2000,SRS378976,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,367368,,,,,,Generic sample from Pelusios castaneus,SAMN01823445,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sideneck turtle,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX209126,SRP017457,Illumina HiSeq 2000,SRS378975,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,142480,,,,,,Generic sample from Sternotherus odoratus,SAMN01823444,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Stinkpot turtle,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX209125,SRP017457,Illumina HiSeq 2000,SRS378974,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,8475,,,,,,Generic sample from Chelydra serpentina,SAMN01823443,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Snapping turtle,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX209124,SRP017457,Illumina HiSeq 2000,SRS378972,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,51853,,,,,,Generic sample from Candoia aspera,SAMN01823442,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Viper boa,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX209123,SRP017457,Illumina HiSeq 2000,SRS378973,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,196253,,,,,,Generic sample from Xenopeltis unicolor,SAMN01823441,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sunbeam Snake,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX209122,SRP017457,Illumina HiSeq 2000,SRS378971,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,8715,,,,,,Generic sample from Agkistrodon piscivorus,SAMN01823440,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX209128,SRP017457,Illumina HiSeq 2000,SRS378977,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,158814,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Terrapene carolina,SAMN01823446,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Box Turtle,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX209127,SRP017457,Illumina HiSeq 2000,SRS378976,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,367368,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Pelusios castaneus,SAMN01823445,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sideneck turtle,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX209126,SRP017457,Illumina HiSeq 2000,SRS378975,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,142480,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Sternotherus odoratus,SAMN01823444,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Stinkpot turtle,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX209125,SRP017457,Illumina HiSeq 2000,SRS378974,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,8475,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Chelydra serpentina,SAMN01823443,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Snapping turtle,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX209124,SRP017457,Illumina HiSeq 2000,SRS378972,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,51853,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Candoia aspera,SAMN01823442,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Viper boa,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX209123,SRP017457,Illumina HiSeq 2000,SRS378973,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,196253,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Xenopeltis unicolor,SAMN01823441,,,,,,,,,,,07/05/2026,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sunbeam Snake,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX209122,SRP017457,Illumina HiSeq 2

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-05-08'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX209128,SRP017457,Illumina HiSeq 2000,SRS378977,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,158814,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Terrapene carolina,SAMN01823446,,,,,,,,,,SAC,2026-05-08,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Box Turtle,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX209127,SRP017457,Illumina HiSeq 2000,SRS378976,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,367368,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Pelusios castaneus,SAMN01823445,,,,,,,,,,SAC,2026-05-08,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sideneck turtle,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX209126,SRP017457,Illumina HiSeq 2000,SRS378975,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,142480,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Sternotherus odoratus,SAMN01823444,,,,,,,,,,SAC,2026-05-08,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Stinkpot turtle,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX209125,SRP017457,Illumina HiSeq 2000,SRS378974,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,8475,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Chelydra serpentina,SAMN01823443,,,,,,,,,,SAC,2026-05-08,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Snapping turtle,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX209124,SRP017457,Illumina HiSeq 2000,SRS378972,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,51853,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Candoia aspera,SAMN01823442,,,,,,,,,,SAC,2026-05-08,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Viper boa,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX209123,SRP017457,Illumina HiSeq 2000,SRS378973,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,196253,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Xenopeltis unicolor,SAMN01823441,,,,,,,,,,SAC,2026-05-08,100bp PE reads were obtained from an Illumina Hi Seq 2000 using TruSeq v3 chemistry. Methods identical to TruSeq_RNA_SamplePrep_Guide_15008136_A were used in library preparation. Reads were barcoded with index adapters.,,Sunbeam Snake,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX209122,SRP01745

#### comments

In [13]:
library.loc[:,'comment'] = 'no PMID'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX209128,SRP017457,Illumina HiSeq 2000,SRS378977,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,158814,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Terrapene carolina,SAMN01823446,,,,,,,no PMID,,,SAC,2026-05-08
1,SRX209127,SRP017457,Illumina HiSeq 2000,SRS378976,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,367368,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Pelusios castaneus,SAMN01823445,,,,,,,no PMID,,,SAC,2026-05-08
2,SRX209126,SRP017457,Illumina HiSeq 2000,SRS378975,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,142480,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Sternotherus odoratus,SAMN01823444,,,,,,,no PMID,,,SAC,2026-05-08
3,SRX209125,SRP017457,Illumina HiSeq 2000,SRS378974,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,8475,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Chelydra serpentina,SAMN01823443,,,,,,,no PMID,,,SAC,2026-05-08
4,SRX209124,SRP017457,Illumina HiSeq 2000,SRS378972,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,51853,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Candoia aspera,SAMN01823442,,,,,,,no PMID,,,SAC,2026-05-08
5,SRX209123,SRP017457,Illumina HiSeq 2000,SRS378973,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,196253,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Xenopeltis unicolor,SAMN01823441,,,,,,,no PMID,,,SAC,2026-05-08
6,SRX209122,SRP017457,Illumina HiSeq 2000,SRS378971,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,8715,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Agkistrodon piscivorus,SAMN01823440,,,,,,,no PMID,,,SAC,2026-05-08
7,SRX209121,SRP017457,Illumina HiSeq 2000,SRS378970,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,231986,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Lasius fuliginosus,SAMN01823439,,,,,,,no PMID,,,SAC,2026-05-08
8,SRX209120,SRP017457,Illumina HiSeq 2000,SRS378969,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,481883,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Eublepharis macularius,SAMN01823438,,,,,,,no PMID,,,SAC,2026-05-08
9,SRX209119,SRP017457,Illumina HiSeq 2000,SRS378968,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,Adult,perfect match,not documented,perfect match,NA,,,155319,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Generic sample from Scincella lateralis,SAMN01823437,,,,,,,no PMID,,,SAC,2026-05-08


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP017457,Liver transcriptomes from reptiles,Liver transcriptomes from reptiles: Thamnophis elegans Thamnophis couchi Alligator mississippiensis Anolis sagrei Elgaria multicarinata Sceloporus undulatus Pogona vitticeps Scincella lateralis Eublepharis macularius Lasius fuliginosus Agkistrodon piscivorus Xenopeltis unicolor Candoia aspera Chelydra serpentina Sternotherus odoratus Pelusios castaneus Terrapene carolina,SRA,,,,,,,PRJNA183121,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

16

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP017457,Liver transcriptomes from reptiles,Liver transcriptomes from reptiles: Thamnophis elegans Thamnophis couchi Alligator mississippiensis Anolis sagrei Elgaria multicarinata Sceloporus undulatus Pogona vitticeps Scincella lateralis Eublepharis macularius Lasius fuliginosus Agkistrodon piscivorus Xenopeltis unicolor Candoia aspera Chelydra serpentina Sternotherus odoratus Pelusios castaneus Terrapene carolina,SRA,total,Bgee 1K,16,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA183121,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
#experiment.loc[:,'reference_url'] = ''
experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP017457,Liver transcriptomes from reptiles,Liver transcriptomes from reptiles: Thamnophis elegans Thamnophis couchi Alligator mississippiensis Anolis sagrei Elgaria multicarinata Sceloporus undulatus Pogona vitticeps Scincella lateralis Eublepharis macularius Lasius fuliginosus Agkistrodon piscivorus Xenopeltis unicolor Candoia aspera Chelydra serpentina Sternotherus odoratus Pelusios castaneus Terrapene carolina,SRA,total,Bgee 1K,16,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA183121,,,,,


#### comments

In [19]:
experiment.loc[:,'comment'] = 'no PMID, no paper'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP017457,Liver transcriptomes from reptiles,Liver transcriptomes from reptiles: Thamnophis elegans Thamnophis couchi Alligator mississippiensis Anolis sagrei Elgaria multicarinata Sceloporus undulatus Pogona vitticeps Scincella lateralis Eublepharis macularius Lasius fuliginosus Agkistrodon piscivorus Xenopeltis unicolor Candoia aspera Chelydra serpentina Sternotherus odoratus Pelusios castaneus Terrapene carolina,SRA,total,Bgee 1K,16,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA183121,,,,,"no PMID, no paper"


#### save complete file

In [20]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [21]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [22]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [ ]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

#### view files

In [ ]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

In [ ]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

### add annotations to git

In [ ]:
! git pull

In [ ]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [ ]:
! git add $git_experiment_path $git_library_path

In [ ]:
! git commit -m $commit_message_exp

In [ ]:
! git push

### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push